In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

# Pengaturan tampilan
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Semua library berhasil di-import!")

Semua library berhasil di-import!


In [11]:
uploaded = files.upload()

file_name = next(iter(uploaded))

print("File berhasil di-upload:", file_name)

Saving data pelanggan.xlsx to data pelanggan (1).xlsx
File berhasil di-upload: data pelanggan (1).xlsx


In [ ]:
df = pd.read_excel(file_name)

print("Dataset berhasil dibaca!")
print("Ukuran dataset:", df.shape)

display(df.head())

In [ ]:
print("===== INFORMASI DATASET =====")

# Jumlah baris dan kolom
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

# Nama kolom
print("\nNama kolom:")
print(df.columns.tolist())

# Informasi tipe data
print("\nInformasi dataset:")
df.info()

# Statistik deskriptif
print("\nStatistik deskriptif:")
display(df.describe(include="all").T)

In [ ]:
missing_analysis = pd.DataFrame({
    "Jumlah Missing": df.isnull().sum(),
    "Persentase Missing (%)":
        (df.isnull().mean() * 100).round(2)
})

display(missing_analysis)

In [ ]:
# Visualisasi missing value

plt.figure(figsize=(12, 5))

sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False,
    cmap="viridis"
)

plt.title("Visualisasi Missing Value Dataset")
plt.xlabel("Kolom")
plt.ylabel("Baris")

plt.show()

In [ ]:
# Duplikasi seluruh baris
duplicate_rows = df.duplicated().sum()

# Duplikasi ID pelanggan
duplicate_ids = df["id_pelanggan"].duplicated().sum()

# Jumlah ID unik
unique_ids = df["id_pelanggan"].nunique()

print("===== DUPLICATE ANALYSIS =====")

print("Duplikasi seluruh baris:", duplicate_rows)

print("ID pelanggan berulang:", duplicate_ids)

print("Jumlah ID unik:", unique_ids)

# Menampilkan ID berulang
duplicate_data = df[
    df.duplicated(
        subset=["id_pelanggan"],
        keep=False
    )
].sort_values("id_pelanggan")

display(duplicate_data.head(20))

In [ ]:
print("===== PEMERIKSAAN NILAI TIDAK VALID =====")

# Umur
print(
    "Umur kurang dari 1:",
    (df["umur"] < 1).sum()
)

print(
    "Umur lebih dari 100:",
    (df["umur"] > 100).sum()
)

# Jumlah transaksi
print(
    "Transaksi negatif:",
    (df["jumlah_transaksi_6bln"] < 0).sum()
)

# Total belanja
print(
    "Total belanja negatif:",
    (df["total_belanja_rp"] < 0).sum()
)

# Hari sejak transaksi terakhir
print(
    "Hari transaksi negatif:",
    (df["hari_sejak_transaksi_terakhir"] < 0).sum()
)

# Persentase promo
promo_invalid = (
    df["persen_pakai_promo"].notna() &
    ~df["persen_pakai_promo"].between(0, 100)
)

print(
    "Persentase promo tidak valid:",
    promo_invalid.sum()
)

# Rating
rating_invalid = (
    df["rating_rata"].notna() &
    ~df["rating_rata"].between(1, 5)
)

print(
    "Rating tidak valid:",
    rating_invalid.sum()
)

In [ ]:
df_clean = df.copy()

# Kolom kategori
kolom_kategori = [
    "kecamatan",
    "channel_utama",
    "metode_bayar",
    "kategori_favorit"
]

# Menyeragamkan huruf dan spasi
for col in kolom_kategori:

    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Menyeragamkan nama kecamatan
df_clean["kecamatan"] = df_clean["kecamatan"].replace({
    "bdg wetan": "bandung wetan",
    "buah batu": "buahbatu"
})

print("Standardisasi kategori berhasil!")

for col in kolom_kategori:
    print(f"\n{col}:")
    print(df_clean[col].value_counts())

In [ ]:
# Mengubah tanggal menjadi datetime
df_clean["tanggal_daftar"] = pd.to_datetime(
    df_clean["tanggal_daftar"],
    errors="coerce"
)

# Kolom numerik
kolom_numerik = [
    "umur",
    "jumlah_transaksi_6bln",
    "total_belanja_rp",
    "hari_sejak_transaksi_terakhir",
    "persen_pakai_promo",
    "rating_rata"
]

# Memastikan tipe data numerik
for col in kolom_numerik:

    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

# Umur di luar rentang 1-100 menjadi missing
df_clean.loc[
    ~df_clean["umur"].between(1, 100),
    "umur"
] = np.nan

# Jumlah transaksi negatif
df_clean.loc[
    df_clean["jumlah_transaksi_6bln"] < 0,
    "jumlah_transaksi_6bln"
] = np.nan

# Total belanja negatif
df_clean.loc[
    df_clean["total_belanja_rp"] < 0,
    "total_belanja_rp"
] = np.nan

# Hari sejak transaksi negatif
df_clean.loc[
    df_clean["hari_sejak_transaksi_terakhir"] < 0,
    "hari_sejak_transaksi_terakhir"
] = np.nan

# Promo di luar 0-100
df_clean.loc[
    ~df_clean["persen_pakai_promo"].between(0, 100),
    "persen_pakai_promo"
] = np.nan

# Rating di luar 1-5
df_clean.loc[
    ~df_clean["rating_rata"].between(1, 5),
    "rating_rata"
] = np.nan

print("Validasi dan perbaikan nilai selesai!")

In [ ]:
# A. Menghapus baris identik
jumlah_duplikat = df_clean.duplicated().sum()

df_clean = df_clean.drop_duplicates().copy()

print("Baris identik yang dihapus:", jumlah_duplikat)

# B. Memeriksa ID pelanggan berulang
print(
    "ID berulang setelah penghapusan:",
    df_clean["id_pelanggan"].duplicated().sum()
)

# C. Menangani ID kosong jika ada
df_clean = df_clean.dropna(
    subset=["id_pelanggan"]
).copy()

# D. Fungsi mengambil modus
def mode_or_first(series):

    mode = series.dropna().mode()

    if not mode.empty:
        return mode.iloc[0]

    non_null = series.dropna()

    if not non_null.empty:
        return non_null.iloc[0]

    return np.nan


# E. Menentukan aturan agregasi
aturan_agregasi = {}

for col in df_clean.columns:

    if col == "id_pelanggan":
        continue

    elif col == "tanggal_daftar":
        aturan_agregasi[col] = "min"

    elif col in kolom_numerik:
        aturan_agregasi[col] = "median"

    else:
        aturan_agregasi[col] = mode_or_first


# F. Menggabungkan data berdasarkan ID
df_clean = (
    df_clean
    .groupby("id_pelanggan", as_index=False)
    .agg(aturan_agregasi)
)

# G. Membulatkan kolom yang seharusnya bilangan bulat
kolom_integer = [
    "jumlah_transaksi_6bln",
    "total_belanja_rp",
    "hari_sejak_transaksi_terakhir",
    "persen_pakai_promo"
]

for col in kolom_integer:

    df_clean[col] = (
        df_clean[col]
        .round()
        .astype("Int64")
    )

print("\nKonsolidasi pelanggan berhasil!")

print("Jumlah baris akhir:", len(df_clean))

print(
    "Jumlah ID unik:",
    df_clean["id_pelanggan"].nunique()
)

In [ ]:
print("=" * 50)
print("FINAL DATA VALIDATION")
print("=" * 50)

print("\nJumlah baris:", df_clean.shape[0])
print("Jumlah kolom:", df_clean.shape[1])

print(
    "Jumlah ID unik:",
    df_clean["id_pelanggan"].nunique()
)

print(
    "ID duplikat:",
    df_clean["id_pelanggan"].duplicated().sum()
)

print(
    "Duplikasi seluruh baris:",
    df_clean.duplicated().sum()
)

print("\nMissing Value:")
display(df_clean.isnull().sum())

print("\nStatistik Deskriptif:")
display(df_clean.describe(include="all").T)

print("\nPemeriksaan rentang umur:")
print(
    df_clean["umur"].min(),
    "-",
    df_clean["umur"].max()
)

print("\nPemeriksaan rentang rating:")
print(
    df_clean["rating_rata"].min(),
    "-",
    df_clean["rating_rata"].max()
)

In [ ]:
kolom_visualisasi = [
    "umur",
    "jumlah_transaksi_6bln",
    "total_belanja_rp",
    "hari_sejak_transaksi_terakhir",
    "persen_pakai_promo",
    "rating_rata"
]

df_clean[kolom_visualisasi].hist(
    figsize=(15, 10),
    bins=30,
    edgecolor="black"
)

plt.suptitle(
    "Distribusi Data Pelanggan Setelah Cleaning",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# VISUALISASI DATA KATEGORIK
# ============================================

kolom_kategori = [
    "kecamatan",
    "channel_utama",
    "metode_bayar",
    "kategori_favorit"
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, ax in zip(kolom_kategori, axes.flatten()):

    order = df_clean[col].value_counts().index

    sns.countplot(
        data=df_clean,
        y=col,
        order=order,
        ax=ax
    )

    ax.set_title(f"Distribusi {col}")

plt.tight_layout()
plt.show()

In [ ]:
# Simpan CSV
df_clean.to_csv(
    "data_pelanggan_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# Simpan Excel
df_clean.to_excel(
    "data_pelanggan_clean.xlsx",
    index=False
)

print("Dataset berhasil disimpan!")

In [ ]:
files.download("data_pelanggan_clean.csv")

files.download("data_pelanggan_clean.xlsx")